In [0]:
%run ../00-common/config

In [0]:
%run ./00_silver_helpers

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

reviews = spark.table(f"{catalog_name}.{bronze_schema}.order_reviews").drop("ingestion_timestamp", "source_file", "batch_id")

# quarantine pa order_id ose pa score
reviews_v1 = split_and_quarantine(
    reviews,
    valid_condition=F.col("order_id").isNotNull() & F.col("review_score").isNotNull(),
    reject_reason="null order_id or score",
    table_name="order_reviews",
    catalog=catalog_name, quarantine_schema=quarantine_schema
)

# konverto tipet (ishin string ne bronze)
reviews_v2 = (reviews_v1
    .withColumn("review_score", F.col("review_score").cast("int"))
    .withColumn("review_creation_date", F.col("review_creation_date").cast("timestamp"))
    .withColumn("review_answer_timestamp", F.col("review_answer_timestamp").cast("timestamp")))

# dedup: per review_id te perseritur, mbaj me te fundit sipas dates
reviews_dedup = (reviews_v2
    .withColumn("rn", F.row_number().over(
        Window.partitionBy("review_id").orderBy(F.desc("review_creation_date"))))
    .filter(F.col("rn") == 1)
    .drop("rn"))

write_to_silver(reviews_dedup, "order_reviews", catalog_name, silver_schema)